# Preparación de datos para predicción (horizonte h=6/12/24)

**Proyecto de Aprendizaje Automático en Series Temporales y Flujos de Datos**

Este cuaderno construye el dataset de *features* que usarán los notebooks de modelado, a partir de `../data/processed/beijing_data_preparado.csv` (salida de `02_preparacion_datos.ipynb`). El horizonte de pronóstico queda fijado en **h=6/12/24 horas**.

Ir variando el número de horas para tener los 3 datasets, dependiendo del límite de predicción que se quiera

In [227]:
NUMERO_DE_HORAS=6

## 1. Carga del dataset preparado

In [228]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/beijing_data_preparado.csv', index_col='datetime', parse_dates=True)
df = df.asfreq('h')
df.drop(columns=["pm2.5_imputado"],inplace=True) #quito la columna de pm2.5_imputado, no la uso para predecir

print("Observaciones:", len(df))
df.head()

Observaciones: 43824


,pm2.5,DEWP,TEMP,PRES,Iws,Is,Ir,viento_NE,viento_NW,viento_SE,viento_cv
datetime,,,,,,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,1.79,0,0,False,True,False,False
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,4.92,0,0,False,True,False,False
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,6.71,0,0,False,True,False,False
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,9.84,0,0,False,True,False,False
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,12.97,0,0,False,True,False,False


## 2. Variables de calendario (codificación cíclica seno/coseno)

Se codifican hora del día, día de la semana y mes como pares seno/coseno para que el modelo vea la naturaleza cíclica de estas variables.

IMPORTANTE: Las variables de calendario se conocen en todo momento y no serán desplazadas con lags.


NOTA: Añadimos dos variables más para mes para capturar la "bimodalidad"?

In [229]:
hora = df.index.hour
dia = df.index.dayofweek
mes = df.index.month

df['cal_sen_hora'] = np.sin(2 * np.pi * hora / 24)
df['cal_cos_hora'] = np.cos(2 * np.pi * hora / 24)

df['cal_sen_dia'] = np.sin(2 * np.pi * dia / 7)
df['cal_cos_dia'] = np.cos(2 * np.pi * dia / 7)

df['cal_sen_mes'] = np.sin(2 * np.pi * (mes - 1) / 12)
df['cal_cos_mes'] = np.cos(2 * np.pi * (mes - 1) / 12)

df[['cal_sen_hora', 'cal_cos_hora', 'cal_sen_dia', 'cal_cos_dia', 'cal_sen_mes', 'cal_cos_mes']].head()

,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes
datetime,,,,,,
2010-01-01 00:00:00,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0
2010-01-01 01:00:00,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0
2010-01-01 02:00:00,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0
2010-01-01 03:00:00,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0
2010-01-01 04:00:00,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0


## 3 - Creación de lags a "NUMERO_DE_HORAS" horas

Para toda exógena que no sea de calendario, la desplazo con un lag de "NUMERO_DE_HORAS" horas. En este caso, para cada valor a predecir (variable y), la información de la exógena que tengo es lo que sucedió "NUMERO_DE_HORAS" horas antes.

Se creará por tanto las variables "var_obs" (variable observada) con dicho lag.

In [230]:
cols_exog_no_cal = [c for c in df.columns if not c.startswith('cal')]

obs = df[cols_exog_no_cal].shift(NUMERO_DE_HORAS).add_suffix(f'_obs_{NUMERO_DE_HORAS}')
df = pd.concat([df, obs], axis=1)

df.filter(like='_obs').head(8)

,pm2.5_obs_6,DEWP_obs_6,TEMP_obs_6,PRES_obs_6,Iws_obs_6,Is_obs_6,Ir_obs_6,viento_NE_obs_6,viento_NW_obs_6,viento_SE_obs_6,viento_cv_obs_6
datetime,,,,,,,,,,,
2010-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 06:00:00,129.0,-21.0,-11.0,1021.0,1.79,0.0,0.0,False,True,False,False
2010-01-01 07:00:00,129.0,-21.0,-12.0,1020.0,4.92,0.0,0.0,False,True,False,False


In [231]:
cols_predictivas = [c for c in df.columns if c.startswith("cal") or "obs" in c]
cols_predictivas

['cal_sen_hora',
 'cal_cos_hora',
 'cal_sen_dia',
 'cal_cos_dia',
 'cal_sen_mes',
 'cal_cos_mes',
 'pm2.5_obs_6',
 'DEWP_obs_6',
 'TEMP_obs_6',
 'PRES_obs_6',
 'Iws_obs_6',
 'Is_obs_6',
 'Ir_obs_6',
 'viento_NE_obs_6',
 'viento_NW_obs_6',
 'viento_SE_obs_6',
 'viento_cv_obs_6']

In [232]:
df_predictivo = df[["pm2.5"]+cols_predictivas]
df_predictivo = df_predictivo.copy()
df_predictivo.head()

,pm2.5,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes,pm2.5_obs_6,DEWP_obs_6,TEMP_obs_6,PRES_obs_6,Iws_obs_6,Is_obs_6,Ir_obs_6,viento_NE_obs_6,viento_NW_obs_6,viento_SE_obs_6,viento_cv_obs_6
datetime,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,129.0,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 01:00:00,129.0,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 02:00:00,129.0,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 03:00:00,129.0,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 04:00:00,129.0,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ahora le meteré distintos retardos de la variable original junto con una variable de tendencia y media movil  

In [233]:
lags_adicionales = [6, 12, 18]  # horas EXTRA por detrás de NUMERO_DE_HORAS
ventanas_roll = [6,12,24]

# Lags adicionales de pm2.5, más antiguos que NUMERO_DE_HORAS
for extra in lags_adicionales:
    h_total = NUMERO_DE_HORAS + extra
    df_predictivo[f'pm2.5_obs_{h_total}'] = df_predictivo['pm2.5'].shift(h_total)

# Tendencia reciente: diferencia entre el lag base y el siguiente lag
df_predictivo[f'pm2.5_trend_{NUMERO_DE_HORAS}_{NUMERO_DE_HORAS + lags_adicionales[0]}'] = (
    df_predictivo[f'pm2.5_obs_{NUMERO_DE_HORAS}'] - df_predictivo[f'pm2.5_obs_{NUMERO_DE_HORAS + lags_adicionales[0]}']
)

# Media/std móvil, calculada hasta t - NUMERO_DE_HORAS (no hasta t)
for ventana_roll in ventanas_roll:
    df_predictivo[f'pm2.5_roll_mean_{ventana_roll}_obs_{NUMERO_DE_HORAS}'] = (
        df_predictivo['pm2.5'].shift(NUMERO_DE_HORAS).rolling(ventana_roll, min_periods=ventana_roll // 2).mean()
    )
    df_predictivo[f'pm2.5_roll_std_{ventana_roll}_obs_{NUMERO_DE_HORAS}'] = (
        df_predictivo['pm2.5'].shift(NUMERO_DE_HORAS).rolling(ventana_roll, min_periods=ventana_roll // 2).std()
    )

Para las variables de temperatura y presión tomaremos la variable de TREND también. Las de viento acumulado no tiene sentido porque ya tienen ese sentido "acumulado"

In [234]:
# Tendencia reciente (6h) de TEMP y PRES: diferencia entre el lag base y 6h más atrás
h_trend_extra = NUMERO_DE_HORAS + lags_adicionales[0]
for var in ['TEMP', 'PRES']:
    obs_extra = df[var].shift(h_trend_extra)
    df_predictivo[f'{var}_trend_{NUMERO_DE_HORAS}_{h_trend_extra}'] = (
        df_predictivo[f'{var}_obs_{NUMERO_DE_HORAS}'] - obs_extra
    )

Finalmente nos quedamos con las filas que no contienen valores nulos añadidos por los lags (las "NUMERO_DE_HORAS" primeras se eliminan)

In [235]:
df_predictivo.isna().sum()

pm2.5                       1463
cal_sen_hora                   0
cal_cos_hora                   0
cal_sen_dia                    0
cal_cos_dia                    0
cal_sen_mes                    0
cal_cos_mes                    0
pm2.5_obs_6                 1469
DEWP_obs_6                     6
TEMP_obs_6                     6
PRES_obs_6                     6
Iws_obs_6                      6
Is_obs_6                       6
Ir_obs_6                       6
viento_NE_obs_6                6
viento_NW_obs_6                6
viento_SE_obs_6                6
viento_cv_obs_6                6
pm2.5_obs_12                1475
pm2.5_obs_18                1481
pm2.5_obs_24                1487
pm2.5_trend_6_12            1738
pm2.5_roll_mean_6_obs_6     1401
pm2.5_roll_std_6_obs_6      1401
pm2.5_roll_mean_12_obs_6    1381
pm2.5_roll_std_12_obs_6     1381
pm2.5_roll_mean_24_obs_6    1359
pm2.5_roll_std_24_obs_6     1359
TEMP_trend_6_12               12
PRES_trend_6_12               12
dtype: int

In [236]:
# Solo exógenas reales: las derivadas de pm2.5 pueden tener NaN aquí para no romper la continuidad del índice (SARIMA)
cols_na = [c for c in df_predictivo.columns if 'pm2.5' not in c]
df_predictivo = df_predictivo.dropna(subset=cols_na)

Cambio los booleanos a Int por si acaso. También, quitaré una de las variables de viento, porque si no, habría colinealidad perfecta (una sobra)

In [237]:
cols_viento = df_predictivo.filter(like='viento').columns
df_predictivo[cols_viento] = df_predictivo[cols_viento].astype(int)

cols_cv = df_predictivo.filter(like='_cv_').columns
df_predictivo = df_predictivo.drop(columns=cols_cv)

In [238]:
df_predictivo.head()

,pm2.5,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes,pm2.5_obs_6,DEWP_obs_6,TEMP_obs_6,...,pm2.5_obs_24,pm2.5_trend_6_12,pm2.5_roll_mean_6_obs_6,pm2.5_roll_std_6_obs_6,pm2.5_roll_mean_12_obs_6,pm2.5_roll_std_12_obs_6,pm2.5_roll_mean_24_obs_6,pm2.5_roll_std_24_obs_6,TEMP_trend_6_12,PRES_trend_6_12
datetime,,,,,,,,,,,,,,,,,,,,,
2010-01-01 12:00:00,129.0,1.224647e-16,-1.000000,-0.433884,-0.900969,0.0,1.0,129.0,-19.0,-9.0,...,NaN,0.0,129.0,0.0,129.0,0.0,NaN,NaN,2.0,-4.0
2010-01-01 13:00:00,129.0,-2.588190e-01,-0.965926,-0.433884,-0.900969,0.0,1.0,129.0,-19.0,-9.0,...,NaN,0.0,129.0,0.0,129.0,0.0,NaN,NaN,3.0,-3.0
2010-01-01 14:00:00,129.0,-5.000000e-01,-0.866025,-0.433884,-0.900969,0.0,1.0,129.0,-19.0,-9.0,...,NaN,0.0,129.0,0.0,129.0,0.0,NaN,NaN,2.0,-2.0
2010-01-01 15:00:00,129.0,-7.071068e-01,-0.707107,-0.433884,-0.900969,0.0,1.0,129.0,-20.0,-8.0,...,NaN,0.0,129.0,0.0,129.0,0.0,NaN,NaN,6.0,-2.0
2010-01-01 16:00:00,129.0,-8.660254e-01,-0.500000,-0.433884,-0.900969,0.0,1.0,129.0,-19.0,-7.0,...,NaN,0.0,129.0,0.0,129.0,0.0,NaN,NaN,5.0,-1.0


In [239]:
df_predictivo.isna().sum()

pm2.5                       1463
cal_sen_hora                   0
cal_cos_hora                   0
cal_sen_dia                    0
cal_cos_dia                    0
cal_sen_mes                    0
cal_cos_mes                    0
pm2.5_obs_6                 1463
DEWP_obs_6                     0
TEMP_obs_6                     0
PRES_obs_6                     0
Iws_obs_6                      0
Is_obs_6                       0
Ir_obs_6                       0
viento_NE_obs_6                0
viento_NW_obs_6                0
viento_SE_obs_6                0
pm2.5_obs_12                1463
pm2.5_obs_18                1469
pm2.5_obs_24                1475
pm2.5_trend_6_12            1726
pm2.5_roll_mean_6_obs_6     1393
pm2.5_roll_std_6_obs_6      1393
pm2.5_roll_mean_12_obs_6    1370
pm2.5_roll_std_12_obs_6     1370
pm2.5_roll_mean_24_obs_6    1347
pm2.5_roll_std_24_obs_6     1347
TEMP_trend_6_12                0
PRES_trend_6_12                0
dtype: int64

In [240]:
df_predictivo.to_csv(f'../data/processed/beijing_data_huecos_{NUMERO_DE_HORAS}h.csv')

### Ahora aquí para preparar los datos para los modelos de machine learning si quitaré todos los nulos (de pm2.5 incluido)

In [241]:
# Para ML: target y todas las features derivadas de pm2.5 completas
cols_pm25 = [c for c in df_predictivo.columns if 'pm2.5' in c]
df_predictivo = df_predictivo.dropna(subset=cols_pm25)

In [242]:
df_predictivo.head()

,pm2.5,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes,pm2.5_obs_6,DEWP_obs_6,TEMP_obs_6,...,pm2.5_obs_24,pm2.5_trend_6_12,pm2.5_roll_mean_6_obs_6,pm2.5_roll_std_6_obs_6,pm2.5_roll_mean_12_obs_6,pm2.5_roll_std_12_obs_6,pm2.5_roll_mean_24_obs_6,pm2.5_roll_std_24_obs_6,TEMP_trend_6_12,PRES_trend_6_12
datetime,,,,,,,,,,,,,,,,,,,,,
2010-01-02 00:00:00,129.0,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0,129.0,-18.0,-5.0,...,129.0,0.0,129.0,0.0,129.0,0.0,129.0,0.0,0.0,1.0
2010-01-02 01:00:00,148.0,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0,129.0,-17.0,-4.0,...,129.0,0.0,129.0,0.0,129.0,0.0,129.0,0.0,-1.0,2.0
2010-01-02 02:00:00,159.0,0.500000,0.866025,-0.974928,-0.222521,0.0,1.0,129.0,-17.0,-5.0,...,129.0,0.0,129.0,0.0,129.0,0.0,129.0,0.0,-3.0,3.0
2010-01-02 03:00:00,181.0,0.707107,0.707107,-0.974928,-0.222521,0.0,1.0,129.0,-17.0,-5.0,...,129.0,0.0,129.0,0.0,129.0,0.0,129.0,0.0,-4.0,4.0
2010-01-02 04:00:00,138.0,0.866025,0.500000,-0.974928,-0.222521,0.0,1.0,129.0,-17.0,-5.0,...,129.0,0.0,129.0,0.0,129.0,0.0,129.0,0.0,-3.0,3.0


In [243]:
df_predictivo.to_csv(f'../data/processed/beijing_data_{NUMERO_DE_HORAS}h.csv')